# 03 - 函数调用深潜 (Function Calling Deep Dive)

## 学习目标

1. **Part 1**: 掌握 JSON Schema 设计原则 —— 如何编写清晰、精确的工具描述
2. **Part 2**: 理解函数路由策略 —— Auto-Routing、Forced-Routing、Pre-Classification
3. **Part 3**: 实现并行函数调用 —— 依赖检测 + asyncio 并行执行
4. **Part 4**: 掌握工具结果处理 —— 错误恢复、重试、截断、格式化

本 Notebook 中的所有代码都是可运行的（使用 Mock LLM 模拟）。

In [ ]:
# 核心导入
import json
import time
import asyncio
import re
from typing import Any, Callable, Optional
from dataclasses import dataclass, field
from enum import Enum
from collections import defaultdict

print("导入完成")

---
## Part 1: JSON Schema 设计原则

### 为什么 Schema 设计如此重要？

工具描述（JSON Schema）是 LLM 理解工具的**唯一途径**。一个设计良好的 Schema：
- 让 LLM 清楚知道何时调用哪个工具
- 减少错误的工具调用
- 提供足够的参数约束以防止无效输入

### 核心原则

1. **清晰的触发条件**：明确说明 "When to use this tool"
2. **严格的参数约束**：enum、min/max、required、format
3. **负面示例**：说明 "Do NOT use this tool when..."
4. **复杂嵌套结构**：支持 object 和 array 嵌套

In [ ]:
# === Part 1: Schema 设计对比 ===

# ===== 反面案例: 设计不良的 Schema =====

BAD_SCHEMA_EXAMPLE = {
    "name": "search",
    "description": "Search for things",  # 太模糊！
    "parameters": {
        "type": "object",
        "properties": {
            "q": {"type": "string"},  # 参数名不直观
            "n": {"type": "integer"},  # 含义不明确
        },
        "required": [],  # 没有 required！
    },
}

# ===== 正面案例: 设计良好的 Schema =====

WELL_DESIGNED_SCHEMAS = {
    "get_weather": {
        "name": "get_weather",
        "description": (
            "获取指定城市的实时天气信息。"
            "【使用场景】用户询问某地天气、气温、是否下雨、穿衣建议等。"
            "【不要使用】当用户询问城市人口、GDP、景点等非天气信息时，"
            "请使用 web_search 而非此工具。"
            "【典型示例】'北京今天天气'、'上海明天会下雨吗'"
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "城市名称，支持中文名（'北京'）和英文名（'Beijing'）",
                    "minLength": 1,
                    "maxLength": 50,
                },
                "date": {
                    "type": "string",
                    "enum": ["today", "tomorrow", "yesterday"],
                    "description": "查询日期，默认为 'today'",
                    "default": "today",
                },
                "units": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "温度单位，默认摄氏度",
                    "default": "celsius",
                },
            },
            "required": ["city"],  # city 是必填的
        },
    },

    "web_search": {
        "name": "web_search",
        "description": (
            "在互联网上搜索信息。适用于获取事实、概念解释、最新资讯等。"
            "【使用场景】用户询问定义、事实、教程、新闻、价格等信息。"
            "【不要使用】用于数学计算（用 calculator）、天气查询（用 get_weather）、"
            "数据库查询（用 database_query）。"
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "搜索关键词，越精准越好。例如 'Python 列表推导式' 而非 'python'",
                    "minLength": 1,
                    "maxLength": 300,
                },
                "num_results": {
                    "type": "integer",
                    "description": "返回结果数量",
                    "minimum": 1,
                    "maximum": 10,
                    "default": 5,
                },
                "language": {
                    "type": "string",
                    "enum": ["zh", "en", "auto"],
                    "description": "搜索结果语言偏好",
                    "default": "auto",
                },
            },
            "required": ["query"],
        },
    },

    "calculator": {
        "name": "calculator",
        "description": (
            "安全的数学表达式计算器。支持基本运算和常用数学函数。"
            "【使用场景】用户需要算术计算、数学表达式求值。"
            "【不要使用】处理文本、查询信息、调用 API。"
            "表达式中有明确的数字和运算符（+、-、*、/）时使用此工具。"
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": (
                        "数学表达式。支持的运算符: +, -, *, /, **, %。"
                        "支持的函数: sqrt, sin, cos, tan, log, abs, round。"
                        "示例: '25 * 4 + 10', 'sqrt(144)', 'sin(pi/2)'"
                    ),
                    "minLength": 1,
                    "maxLength": 500,
                },
            },
            "required": ["expression"],
        },
    },

    # 复杂嵌套 Schema 示例
    "send_email": {
        "name": "send_email",
        "description": (
            "发送电子邮件。支持多个收件人、抄送、附件。"
            "【使用场景】用户明确要求发送邮件。"
            "【不要使用】用户只是在讨论邮件内容但未要求发送。"
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "to": {
                    "type": "array",
                    "description": "收件人列表",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {
                                "type": "string",
                                "description": "收件人姓名",
                            },
                            "email": {
                                "type": "string",
                                "format": "email",
                                "description": "收件人邮箱地址",
                            },
                        },
                        "required": ["email"],
                    },
                    "minItems": 1,
                    "maxItems": 50,
                },
                "subject": {
                    "type": "string",
                    "description": "邮件主题",
                    "minLength": 1,
                    "maxLength": 200,
                },
                "body": {
                    "type": "string",
                    "description": "邮件正文（支持 Markdown）",
                    "maxLength": 10000,
                },
                "cc": {
                    "type": "array",
                    "description": "抄送列表",
                    "items": {"type": "string", "format": "email"},
                },
                "attachments": {
                    "type": "array",
                    "description": "附件列表",
                    "items": {
                        "type": "object",
                        "properties": {
                            "filename": {"type": "string"},
                            "content": {"type": "string", "description": "Base64 编码的附件内容"},
                            "mime_type": {"type": "string", "default": "application/octet-stream"},
                        },
                        "required": ["filename", "content"],
                    },
                },
            },
            "required": ["to", "subject", "body"],
        },
    },
}

print("Schema 示例已定义:")
for name in WELL_DESIGNED_SCHEMAS:
    print(f"  - {name}")
print(f"\n反面案例 (BAD_SCHEMA_EXAMPLE) 的问题:")
print(f"  1. 描述太模糊（'Search for things'）")
print(f"  2. 参数名不直观（'q' 和 'n'）")
print(f"  3. 缺少 required 字段")
print(f"  4. 缺少使用/不使用场景说明")

### Schema 设计检查清单 + Before/After 对比

In [ ]:
class SchemaValidator:
    """Schema 设计质量检查器。

    自动检查工具 Schema 是否符合最佳实践。
    每个检查项输出通过/不通过和建议。
    """

    # 检查清单
    CHECKLIST = [
        ("has_name", "有 name 字段"),
        ("has_description", "有 description 字段"),
        ("desc_has_when_to_use", "描述中包含【使用场景】或触发条件"),
        ("desc_has_when_not_to_use", "描述中包含【不要使用】或排除条件"),
        ("desc_has_examples", "描述中包含示例"),
        ("desc_min_length", "描述长度 >= 50 字符"),
        ("has_parameters", "有 parameters 定义"),
        ("params_is_object", "parameters.type 为 'object'"),
        ("has_required", "有 required 字段"),
        ("required_non_empty", "required 至少包含一个字段（对关键工具）"),
        ("params_have_descriptions", "每个参数都有 description"),
        ("params_have_types", "每个参数都指定了 type"),
        ("has_input_constraints", "参数有约束（enum/min/max/format）"),
    ]

    @classmethod
    def validate(cls, schema: dict) -> dict:
        """验证一个 Schema 并返回详细报告。

        Args:
            schema: 工具的 JSON Schema

        Returns:
            {"score": 0-100, "passed": [...], "failed": [...], "suggestions": [...]}
        """
        passed = []
        failed = []
        suggestions = []

        # 1. name
        if "name" in schema:
            passed.append("has_name")
        else:
            failed.append("has_name")
            suggestions.append("添加 'name' 字段")

        # 2-6. description
        desc = schema.get("description", "")
        if desc:
            passed.append("has_description")
            if len(desc) >= 50:
                passed.append("desc_min_length")
            else:
                failed.append("desc_min_length")
                suggestions.append(f"描述太短（{len(desc)} 字符），建议 >= 50 字符")

            if any(kw in desc for kw in ["使用场景", "触发条件", "When to use", "用于"]):
                passed.append("desc_has_when_to_use")
            else:
                failed.append("desc_has_when_to_use")
                suggestions.append("在描述中添加【使用场景】说明")

            if any(kw in desc for kw in ["不要使用", "不要用", "Do NOT", "禁止"]):
                passed.append("desc_has_when_not_to_use")
            else:
                failed.append("desc_has_when_not_to_use")
                suggestions.append("在描述中添加【不要使用】说明，帮助 LLM 排除错误的工具")

            if any(kw in desc for kw in ["示例", "例如", "Example", "e.g."]):
                passed.append("desc_has_examples")
            else:
                failed.append("desc_has_examples")
                suggestions.append("在描述中加入使用示例")
        else:
            failed.extend(["has_description", "desc_min_length", "desc_has_when_to_use",
                          "desc_has_when_not_to_use", "desc_has_examples"])
            suggestions.append("必须添加 'description' 字段")

        # 7-13. parameters
        params = schema.get("parameters", {})
        if params:
            passed.append("has_parameters")

            if params.get("type") == "object":
                passed.append("params_is_object")
            else:
                failed.append("params_is_object")
                suggestions.append("parameters.type 应为 'object'")

            if "required" in params:
                passed.append("has_required")
                if len(params["required"]) > 0:
                    passed.append("required_non_empty")
                else:
                    failed.append("required_non_empty")
            else:
                failed.append("has_required")
                suggestions.append("添加 'required' 字段，标注必填参数")

            # 检查每个参数
            props = params.get("properties", {})
            all_have_desc = True
            all_have_type = True
            has_constraints = False

            for param_name, param_schema in props.items():
                if "description" not in param_schema:
                    all_have_desc = False
                    suggestions.append(f"参数 '{param_name}' 缺少 description")
                if "type" not in param_schema:
                    all_have_type = False
                    suggestions.append(f"参数 '{param_name}' 缺少 type")
                if any(k in param_schema for k in ["enum", "minimum", "maximum",
                    "minLength", "maxLength", "format", "pattern"]):
                    has_constraints = True

            if all_have_desc:
                passed.append("params_have_descriptions")
            else:
                failed.append("params_have_descriptions")

            if all_have_type:
                passed.append("params_have_types")
            else:
                failed.append("params_have_types")

            if has_constraints:
                passed.append("has_input_constraints")
            else:
                failed.append("has_input_constraints")
                suggestions.append("为参数添加约束（enum/min/max 等）减少无效调用")
        else:
            failed.extend(["has_parameters", "params_is_object", "has_required",
                          "required_non_empty", "params_have_descriptions",
                          "params_have_types", "has_input_constraints"])
            suggestions.append("必须添加 'parameters' 定义")

        total = len(cls.CHECKLIST)
        passed_count = len([c for c, _ in cls.CHECKLIST if c[0] in passed])
        score = int((passed_count / total) * 100)

        return {
            "score": score,
            "passed": passed,
            "failed": failed,
            "suggestions": suggestions,
        }

    @classmethod
    def print_report(cls, schema_name: str, schema: dict):
        """打印友好的验证报告。"""
        result = cls.validate(schema)
        print(f"\n{'='*50}")
        print(f"  Schema 质量报告: {schema_name}")
        print(f"{'='*50}")
        print(f"  得分: {result['score']}/100")
        print(f"  通过: {len(result['passed'])}/{len(cls.CHECKLIST)}")
        print(f"  未通过: {len(result['failed'])}/{len(cls.CHECKLIST)}")
        if result['failed']:
            print(f"\n  未通过的检查项:")
            for check_name, check_desc in cls.CHECKLIST:
                if check_name in result['failed']:
                    print(f"    - {check_desc}")
        if result['suggestions']:
            print(f"\n  改进建议:")
            for i, sug in enumerate(result['suggestions'], 1):
                print(f"    {i}. {sug}")
        return result


# 验证良好设计的 Schema
SchemaValidator.print_report(
    "get_weather（良好设计）",
    WELL_DESIGNED_SCHEMAS["get_weather"]
)

# 验证不良设计的 Schema
SchemaValidator.print_report(
    "search（不良设计）",
    BAD_SCHEMA_EXAMPLE
)

---
## Part 2: 函数路由策略

当 Agent 有多个工具时，需要决定调用哪个工具。三种路由策略：

1. **Auto-Routing（自动路由）**：将全部工具列表给模型，让模型自行判断
2. **Forced-Routing（强制路由）**：通过 `tool_choice` 参数强制指定工具
3. **Pre-Classification（预分类路由）**：先用规则/轻量模型分类，再提供相关工具子集

下面实现并比较这三种策略。

In [ ]:
# === Part 2: 三种路由策略实现 ===

class ToolRouter:
    """工具路由器：实现三种不同的路由策略。"""

    # 模拟的工具和其适用领域
    TOOL_CATEGORIES = {
        "calculator": "math",
        "get_weather": "weather",
        "web_search": "search",
        "database_query": "database",
        "file_reader": "file",
        "send_email": "communication",
        "translate": "language",
        "get_time": "utility",
    }

    def __init__(self):
        self.call_history = []

    # ===== 策略 1: Auto-Routing =====

    def auto_route(self, task: str, tools_schema: list[dict],
                   model_response: str) -> str:
        """自动路由：模型自主决定调用哪个工具。

        这是最灵活的方案：将所有工具的 Schema 都提供给模型，
        让模型根据用户任务自行选择最合适的工具。

        优点:
        - 最灵活，模型可以自主判断
        - 不需要额外的分类步骤
        - 适用于工具数量不多（< 15 个）的场景

        缺点:
        - 工具太多时模型容易混淆
        - Prompt 长度随工具数量线性增长
        - 模型可能选择错误的工具

        Args:
            task: 用户任务
            tools_schema: 所有工具的 Schema 列表
            model_response: 模型原始响应（模拟）

        Returns:
            路由策略分析报告
        """
        self.call_history.append(("auto", task))

        # 模拟模型的选择过程
        # 实际使用中，这里会调用真实的 LLM API
        selected = self._simulate_tool_selection(task, list(self.TOOL_CATEGORIES.keys()))

        return (
            f"[Auto-Routing]\n"
            f"  可用工具: {len(tools_schema)} 个\n"
            f"  任务分析: {task[:60]}...\n"
            f"  模型选择: {selected}\n"
            f"  Prompt Token 估算: {len(tools_schema) * 120} tokens（工具描述）\n"
            f"  决策时间: 完全由模型推理（~1-3秒）"
        )

    # ===== 策略 2: Forced-Routing =====

    def forced_route(self, task: str, forced_tool: str) -> str:
        """强制路由：用 tool_choice 参数锁定特定工具。

        适用于明确知道应该使用哪个工具的场景。
        通过 tool_choice={"type": "function", "function": {"name": "X"}}
        强制模型必须调用指定工具。

        优点:
        - 确定性最高，绝对不会调用错误的工具
        - Token 消耗最小（只需提供目标工具的 Schema）
        - 响应速度最快

        缺点:
        - 需要预先知道用哪个工具
        - 如果用户的意图被误分类，无法纠正
        - 不适合多步骤任务

        Args:
            task: 用户任务
            forced_tool: 强制使用的工具名称

        Returns:
            路由策略分析报告
        """
        self.call_history.append(("forced", task, forced_tool))

        return (
            f"[Forced-Routing]\n"
            f"  强制工具: {forced_tool}\n"
            f"  任务: {task[:60]}...\n"
            f"  tool_choice: {{'type': 'function', 'function': {{'name': '{forced_tool}'}}}}\n"
            f"  Prompt Token 估算: ~150 tokens（仅目标工具描述）\n"
            f"  决策时间: 无额外推理（直接跳转）"
        )

    # ===== 策略 3: Pre-Classification =====

    def pre_classify_route(self, task: str) -> str:
        """预分类路由：先用分类器确定工具类别，再提供相关工具。

        两阶段流程:
        1. 分类阶段：用规则或轻量模型确定任务类型
        2. 路由阶段：只给 LLM 提供该类别的工具子集

        优点:
        - 减少 LLM 选择的候选工具数量
        - 分类器可以很快（规则匹配）或很便宜（小模型）
        - 分类 + 选择的组合正确率通常高于直接选择

        缺点:
        - 多了一次分类调用
        - 分类错误会导致无法纠正
        - 需要维护分类规则或训练分类模型

        Args:
            task: 用户任务

        Returns:
            路由策略分析报告
        """
        self.call_history.append(("pre_classify", task))

        # 阶段 1: 分类（快速、低成本）
        category = self._classify_task(task)
        category_tools = self._get_tools_for_category(category)

        # 阶段 2: 模型在相关工具中选择
        selected = self._simulate_tool_selection(task, category_tools)

        return (
            f"[Pre-Classification Routing]\n"
            f"  阶段1-分类: 任务类别 = {category}（规则匹配，<1ms）\n"
            f"  阶段2-选择: 候选工具 = {category_tools}（从 {len(self.TOOL_CATEGORIES)} 个缩减到 {len(category_tools)} 个）\n"
            f"  最终选择: {selected}\n"
            f"  Prompt Token 估算: {len(category_tools) * 120} tokens\n"
            f"  决策时间: <1ms（分类）+ ~0.5-1.5s（模型选择）"
        )

    def _classify_task(self, task: str) -> str:
        """基于规则的任务分类。

        这是模拟的分类器。实际使用中可以是:
        - 规则匹配（快速，适合简单场景）
        - 轻量模型如 DistilBERT（平衡速度和质量）
        - 嵌入相似度匹配（适合大量类别）
        """
        task_lower = task.lower()

        rules = [
            ("math", ["计算", "算", "+", "-", "*", "/", "等于", "求值",
                       "calculate", "compute", "sqrt"]),
            ("weather", ["天气", "气温", "温度", "下雨", "晴", "阴",
                         "weather", "temperature", "rain"]),
            ("database", ["员工", "部门", "工资", "薪资", "employee",
                          "salary", "department", "SQL"]),
            ("file", ["文件", "读取", "打开", "file", "read"]),
            ("language", ["翻译", "translate", "译"]),
            ("communication", ["邮件", "发送", "email", "send"]),
            ("utility", ["时间", "日期", "time", "date", "几点", "星期"]),
        ]

        for category, keywords in rules:
            for kw in keywords:
                if kw in task_lower:
                    return category

        return "search"  # 默认类别

    def _get_tools_for_category(self, category: str) -> list[str]:
        """获取某类别相关的工具列表。"""
        mapping = {
            "math": ["calculator"],
            "weather": ["get_weather"],
            "search": ["web_search"],
            "database": ["database_query"],
            "file": ["file_reader"],
            "language": ["translate", "web_search"],
            "communication": ["send_email"],
            "utility": ["get_time", "web_search"],
        }
        return mapping.get(category, ["web_search"])

    def _simulate_tool_selection(self, task: str,
                                  available_tools: list[str]) -> str:
        """模拟模型选择工具。

        实际中这是 LLM 调用。这里用规则模拟各种策略的效果。
        """
        task_lower = task.lower()

        # 优先级规则
        priority = [
            ("math", "calculator"),
            ("weather", "get_weather"),
            ("database", "database_query"),
            ("file", "file_reader"),
            ("language", "translate"),
            ("communication", "send_email"),
            ("utility", "get_time"),
        ]

        for category, tool in priority:
            if tool in available_tools and self._classify_task(task) == category:
                return tool

        return available_tools[0] if available_tools else "web_search"


# 创建路由器并测试
router = ToolRouter()

print("=" * 60)
print("  策略对比测试")
print("=" * 60)

test_tasks = [
    "计算 25 * 4 + 10",
    "查询北京天气",
    "技术部员工平均工资",
]

all_tools_schema = [{"name": name} for name in ToolRouter.TOOL_CATEGORIES.keys()]

for task in test_tasks:
    print(f"\n{'─' * 50}")
    print(f"  任务: {task}")
    print(f"{'─' * 50}")
    print(router.auto_route(task, all_tools_schema, ""))
    print()
    print(router.forced_route(task, router._simulate_tool_selection(
        task, list(ToolRouter.TOOL_CATEGORIES.keys()))))
    print()
    print(router.pre_classify_route(task))

### 路由策略基准测试 (Benchmark)

In [ ]:
# === 路由策略准确率基准测试 ===

def run_routing_benchmark():
    """比较三种路由策略在测试集上的表现。"""

    # 测试数据: (任务, 正确工具)
    test_cases = [
        ("计算 100 / 5", "calculator"),
        ("What is 25 * 4?", "calculator"),
        ("今天北京的天气怎么样", "get_weather"),
        ("上海明天会下雨吗", "get_weather"),
        ("查询故宫门票价格", "web_search"),
        ("Python 是什么", "web_search"),
        ("技术部员工的平均薪资", "database_query"),
        ("读取 README.md 文件", "file_reader"),
        ("翻译 'hello' 到中文", "translate"),
        ("现在几点了", "get_time"),
        ("发送一封邮件给张三", "send_email"),
        ("帮我搜索一下巴黎的景点", "web_search"),
        ("1024 + 2048 等于多少", "calculator"),
        ("广州气温多少度", "get_weather"),
        ("查询所有员工列表", "database_query"),
    ]

    tools = list(ToolRouter.TOOL_CATEGORIES.keys())

    results = {
        "auto": {"correct": 0, "total": 0, "errors": []},
        "forced": {"correct": 0, "total": 0, "errors": []},
        "pre_classify": {"correct": 0, "total": 0, "errors": []},
    }

    test_router = ToolRouter()

    for task, expected in test_cases:
        # Auto-Routing
        auto_selected = test_router._simulate_tool_selection(task, tools)
        results["auto"]["total"] += 1
        if auto_selected == expected:
            results["auto"]["correct"] += 1
        else:
            results["auto"]["errors"].append((task, expected, auto_selected))

        # Forced-Routing（假设我们知道正确的工具）
        # 在实际中，forced 路由的正确性取决于外部决策
        forced_selected = expected  # 假设外部决策总是正确
        results["forced"]["total"] += 1
        if forced_selected == expected:
            results["forced"]["correct"] += 1

        # Pre-Classification
        pre_selected = test_router._simulate_tool_selection(
            task, test_router._get_tools_for_category(
                test_router._classify_task(task)))
        results["pre_classify"]["total"] += 1
        if pre_selected == expected:
            results["pre_classify"]["correct"] += 1
        else:
            results["pre_classify"]["errors"].append(
                (task, expected, pre_selected))

    # 打印报告
    print("\n" + "=" * 60)
    print("  路由策略基准测试报告")
    print("=" * 60)

    for strategy, data in results.items():
        accuracy = (data["correct"] / data["total"] * 100) if data["total"] > 0 else 0
        strategy_names = {
            "auto": "Auto-Routing（自动路由）",
            "forced": "Forced-Routing（强制路由）",
            "pre_classify": "Pre-Classification（预分类）",
        }
        print(f"\n  {strategy_names[strategy]}:")
        print(f"    准确率: {data['correct']}/{data['total']} = {accuracy:.1f}%")
        print(f"    平均 Token: {_estimate_tokens(strategy, tools)}")
        print(f"    平均延迟: {_estimate_latency(strategy)}")
        if data["errors"]:
            print(f"    错误案例:")
            for task, expected, got in data["errors"]:
                print(f"      '{task}' → 预期 {expected}，实际 {got}")


def _estimate_tokens(strategy: str, tools: list[str]) -> str:
    """估算各策略的 Token 消耗。"""
    estimates = {
        "auto": f"~{len(tools) * 120} tokens（所有工具描述）",
        "forced": "~150 tokens（仅目标工具描述）",
        "pre_classify": "~250-400 tokens（分类 + 部分工具描述）",
    }
    return estimates.get(strategy, "N/A")


def _estimate_latency(strategy: str) -> str:
    """估算各策略的延迟。"""
    estimates = {
        "auto": "~1-3秒（模型推理选择）",
        "forced": "~0.5-1秒（跳过选择）",
        "pre_classify": "~0.5-2秒（<1ms分类 + 模型选择）",
    }
    return estimates.get(strategy, "N/A")


run_routing_benchmark()

---
## Part 3: 并行函数调用

当多个工具调用之间没有依赖关系时，可以并行执行以节省时间。
关键挑战：如何检测依赖关系，哪些调用可以并行，哪些必须串行。

In [ ]:
# === Part 3: 并行函数调用 ===

@dataclass
class ToolCall:
    """表示一个工具调用。"""
    id: str                          # 唯一标识
    name: str                        # 工具名称
    params: dict                     # 参数
    depends_on: list[str] = field(default_factory=list)  # 依赖的工具调用 ID
    result: Any = None               # 执行结果


class DependencyDetector:
    """检测工具调用之间的依赖关系。

    依赖类型:
    1. 参数引用: call_B.params.x 引用了 call_A.result.field
    2. 数据依赖: call_B 需要 call_A 的输出数据
    3. 时序依赖: call_B 必须在 call_A 之后执行

    检测方法:
    - 分析参数中是否包含对之前调用结果的引用
    - 检查参数中的占位符: $call_id.result.field
    """

    @staticmethod
    def detect_dependencies(calls: list[ToolCall]) -> list[ToolCall]:
        """检测并标注每个调用的依赖关系。

        Args:
            calls: 工具调用列表

        Returns:
            标注了依赖关系后的调用列表
        """
        for i, call in enumerate(calls):
            # 检查参数中是否引用了之前的调用
            for param_value in call.params.values():
                if isinstance(param_value, str):
                    # 查找 $call_id 模式
                    refs = re.findall(r'\$(call_\d+)\.', param_value)
                    for ref in refs:
                        if ref not in call.depends_on:
                            call.depends_on.append(ref)

        return calls

    @staticmethod
    def get_execution_groups(calls: list[ToolCall]) -> list[list[ToolCall]]:
        """将调用分组为可并行执行的批次。

        算法:
        1. 第 0 组: 所有无依赖的调用（可完全并行）
        2. 第 1 组: 依赖仅在第 0 组中的调用
        3. 第 n 组: 依赖仅在前 n-1 组中的调用

        Returns:
            分组后的调用批次列表
        """
        remaining = list(calls)
        completed_ids: set[str] = set()
        groups = []

        while remaining:
            current_group = []
            still_remaining = []

            for call in remaining:
                # 检查所有依赖是否已完成
                if all(dep in completed_ids for dep in call.depends_on):
                    current_group.append(call)
                else:
                    still_remaining.append(call)

            if not current_group:
                # 死锁：所有剩余的调用都有未解决的依赖
                # 说明参数中引用了不存在的调用
                print(f"[警告] 检测到未解决的依赖: "
                      f"{[(c.id, c.depends_on) for c in remaining]}")
                break

            groups.append(current_group)
            completed_ids.update(c.id for c in current_group)
            remaining = still_remaining

        return groups


class ParallelExecutor:
    """并行工具调用执行器。

    执行策略:
    1. 检测依赖关系
    2. 分组为执行批次
    3. 批次内并行（asyncio.gather）
    4. 批次间串行（等待前一批次完成）
    """

    def __init__(self, tools: dict):
        self.tools = tools

    async def execute_parallel(self, calls: list[ToolCall]) -> list[ToolCall]:
        """并行执行工具调用。

        Args:
            calls: 工具调用列表

        Returns:
            填充了 result 的调用列表
        """
        # 步骤 1: 检测依赖
        calls = DependencyDetector.detect_dependencies(calls)

        # 步骤 2: 分组
        groups = DependencyDetector.get_execution_groups(calls)

        print(f"[并行执行计划]")
        for i, group in enumerate(groups):
            names = [c.name for c in group]
            print(f"  Batch {i}: {names} (并行执行)")

        # 步骤 3: 按批次执行
        all_results: dict[str, Any] = {}  # call_id → result
        start_time = time.time()

        for batch_idx, group in enumerate(groups):
            print(f"\n[Batch {batch_idx}] 开始执行 {len(group)} 个调用...")
            batch_start = time.time()

            # 解析依赖的参数引用
            for call in group:
                resolved_params = self._resolve_params(
                    call.params, all_results)
                call.params = resolved_params

            # 并行执行
            tasks = [self._execute_one(call) for call in group]
            results = await asyncio.gather(*tasks, return_exceptions=True)

            for call, result in zip(group, results):
                if isinstance(result, Exception):
                    call.result = f"错误: {str(result)}"
                else:
                    call.result = result
                all_results[call.id] = call.result

            batch_time = time.time() - batch_start
            print(f"[Batch {batch_idx}] 完成 ({batch_time:.2f}s)")

        total_time = time.time() - start_time
        print(f"\n[总计] {len(calls)} 个调用，{len(groups)} 个批次，总时间 {total_time:.2f}s")

        return calls

    async def _execute_one(self, call: ToolCall) -> str:
        """执行单个工具调用（模拟）。"""
        # 模拟工具执行耗时
        await asyncio.sleep(0.3 + (hash(call.name) % 5) * 0.1)

        if call.name not in self.tools:
            return f"错误: 工具 '{call.name}' 不存在"

        try:
            tool_func = self.tools[call.name]
            result = tool_func(**call.params)
            return str(result)
        except Exception as e:
            return f"错误: {str(e)}"

    def _resolve_params(self, params: dict,
                         results: dict[str, Any]) -> dict:
        """解析参数中的依赖引用。

        例如: {"city": "$call_0.city_name"} →
               {"city": results["call_0"].city_name}
        """
        resolved = {}
        for key, value in params.items():
            if isinstance(value, str):
                # 查找 $call_id.field 模式
                ref_match = re.match(r'^\$(call_\d+)\.(.+)$', value)
                if ref_match:
                    call_id = ref_match.group(1)
                    field = ref_match.group(2)
                    if call_id in results:
                        # 尝试从结果中提取字段
                        resolved[key] = self._extract_field(
                            results[call_id], field)
                    else:
                        resolved[key] = value  # 保留原值
                else:
                    resolved[key] = value
            else:
                resolved[key] = value
        return resolved

    def _extract_field(self, result: Any, field: str) -> str:
        """从结果中提取字段值。"""
        try:
            if isinstance(result, str):
                result_json = json.loads(result)
            else:
                result_json = result

            if isinstance(result_json, dict):
                return str(result_json.get(field, ""))
            return str(result)
        except (json.JSONDecodeError, TypeError):
            return str(result)


# Mock 工具函数
def mock_weather(city: str, date: str = "today") -> str:
    return json.dumps({"city": city, "temperature": 22, "condition": "晴天"})

def mock_search(query: str) -> str:
    return json.dumps({"results": [f"关于 {query} 的搜索结果"]})

def mock_calculator_simple(expression: str) -> str:
    # 安全求值
    import ast, operator as op
    try:
        allowed = {ast.Add: op.add, ast.Sub: op.sub,
                   ast.Mult: op.mul, ast.Div: op.truediv}
        tree = ast.parse(expression, mode="eval")
        def _eval(n):
            if isinstance(n, ast.Constant): return n.value
            if isinstance(n, ast.BinOp): return allowed[type(n.op)](_eval(n.left), _eval(n.right))
            if isinstance(n, ast.Expression): return _eval(n.body)
            raise ValueError("不支持")
        return str(_eval(tree))
    except Exception as e:
        return f"错误: {e}"


TOOLS_POOL = {
    "get_weather": mock_weather,
    "web_search": mock_search,
    "calculator": mock_calculator_simple,
}

print("并行执行器已准备好，工具池:", list(TOOLS_POOL.keys()))

### 演示 1: 独立工具调用（完全并行）

场景：同时查询三个城市的天气 —— 它们之间没有依赖关系，可以完全并行。

In [ ]:
# === 演示 1: 无依赖，完全并行 ===

async def demo_independent_calls():
    """演示完全独立的并行调用。

    三个天气查询之间没有任何依赖关系，
    可以同时发起，获得最大加速。
    """
    print("\n" + "=" * 60)
    print("  演示 1: 独立工具调用（完全并行）")
    print("=" * 60)

    calls = [
        ToolCall(id="call_0", name="get_weather",
                 params={"city": "北京", "date": "today"}),
        ToolCall(id="call_1", name="get_weather",
                 params={"city": "上海", "date": "today"}),
        ToolCall(id="call_2", name="get_weather",
                 params={"city": "广州", "date": "today"}),
    ]

    executor = ParallelExecutor(TOOLS_POOL)

    # 并行执行
    start = time.time()
    results = await executor.execute_parallel(calls)
    parallel_time = time.time() - start

    # 串行执行对比
    print("\n[对比] 串行执行模拟:")
    serial_time = 0.3 * 3 + 0.15 * 3  # 模拟串行时间
    print(f"  并行执行时间: {parallel_time:.2f}s")
    print(f"  串行执行时间（估算）: {serial_time:.2f}s")
    print(f"  加速比: {serial_time/parallel_time:.2f}x")

    print("\n[结果]")
    for call in results:
        print(f"  {call.id} ({call.name}): {call.result}")

    return results

# 运行演示
await demo_independent_calls()

### 演示 2: 有依赖的工具调用（部分并行）

场景：先搜索城市信息，再根据搜索结果中的城市名查询天气。这是典型的链式依赖。

In [ ]:
# === 演示 2: 有依赖关系的调用（部分并行）===

async def demo_dependent_calls():
    """演示有依赖关系的调用。

    call_1 依赖 call_0 的结果（需要先搜索城市，再查天气），
    但 call_2 是独立的计算，可以和 call_0 并行。

    执行计划:
      Batch 0: [call_0 (搜索), call_2 (计算)] ← 并行
      Batch 1: [call_1 (天气)] ← 等 Batch 0 完成后执行
    """
    print("\n" + "=" * 60)
    print("  演示 2: 有依赖的工具调用（部分并行）")
    print("=" * 60)

    calls = [
        ToolCall(id="call_0", name="web_search",
                 params={"query": "法国首都是哪个城市"}),
        ToolCall(id="call_1", name="get_weather",
                 params={"city": "$call_0.result", "date": "today"},
                 depends_on=["call_0"]),  # 依赖 call_0 的结果
        ToolCall(id="call_2", name="calculator",
                 params={"expression": "100 * 2 + 50"}),
    ]

    executor = ParallelExecutor(TOOLS_POOL)
    results = await executor.execute_parallel(calls)

    print("\n[结果]")
    for call in results:
        print(f"  {call.id} ({call.name}): {call.result}")

    return results

await demo_dependent_calls()

---
## Part 4: 工具结果处理

工具调用完成后，如何处理返回结果是一个关键问题。
本节涵盖四个方面：
1. 结构化错误响应
2. 重试逻辑
3. 结果截断
4. 格式化为 Observation

In [ ]:
# === Part 4: 工具结果处理 ===

class ToolResultProcessor:
    """处理工具返回结果的标准流程。

    处理管道:
    1. 检查结果结构（是否为错误响应）
    2. 对可重试错误进行智能重试
    3. 截断过长的结果
    4. 格式化为模型可理解的 Observation 文本
    """

    def __init__(self, max_result_length: int = 2000,
                 max_retries: int = 2):
        self.max_result_length = max_result_length
        self.max_retries = max_retries

    def process(self, tool_name: str, params: dict,
                raw_result: str) -> str:
        """完整的工具结果处理管道。

        Args:
            tool_name: 工具名称
            params: 使用的参数
            raw_result: 工具返回的原始结果

        Returns:
            格式化后的 Observation 文本
        """
        # 步骤 1: 解析结果
        parsed = self._parse_result(raw_result)

        # 步骤 2: 检查错误
        if self._is_error(parsed):
            return self._format_error_as_observation(
                tool_name, params, parsed)

        # 步骤 3: 截断过长结果
        truncated = self._truncate_if_needed(parsed)

        # 步骤 4: 格式化为 Observation
        observation = self._format_success_observation(
            tool_name, truncated)

        return observation

    def _parse_result(self, raw_result: str) -> dict:
        """解析原始结果为结构化数据。"""
        try:
            result = json.loads(raw_result)
            if isinstance(result, dict):
                return result
        except (json.JSONDecodeError, TypeError):
            pass

        # 非 JSON 结果包装为 success
        return {
            "success": True,
            "data": raw_result,
            "raw": True,
        }

    def _is_error(self, result: dict) -> bool:
        """检查结果是否为错误。"""
        if not result.get("success", True):
            return True
        if "error" in result and "error_type" in result:
            return True
        return False

    def _format_error_as_observation(self, tool_name: str,
                                      params: dict,
                                      error: dict) -> str:
        """将工具错误格式化为 LLM 可理解的 Observation。

        关键原则：
        1. 明确告诉 LLM 发生了什么错误
        2. 提供修复建议
        3. 如果可重试，告诉 LLM 如何调整参数
        4. 如果不可重试，建议 LLM 尝试其他方案
        """
        error_type = error.get("error_type", "unknown")
        error_msg = error.get("error", "未知错误")
        suggestion = error.get("suggestion", "")
        retryable = error.get("retryable", False)

        lines = [
            f"[工具调用失败] '{tool_name}' 返回错误",
            f"错误类型: {error_type}",
            f"错误信息: {error_msg}",
        ]

        if suggestion:
            lines.append(f"修复建议: {suggestion}")

        if retryable:
            lines.append(
                f"此错误可以重试。请调整参数（原参数: {params}）后重新调用。"
            )
        else:
            lines.append(
                f"此错误不可重试。建议:\n"
                f"  - 尝试使用其他工具获取相同信息\n"
                f"  - 如果已有足够信息，直接回答用户问题\n"
                f"  - 向用户说明无法完成的原因"
            )

        return "\n".join(lines)

    def _format_success_observation(self, tool_name: str,
                                     result: dict) -> str:
        """将成功结果格式化为 Observation。"""
        if result.get("raw"):
            data = result.get("data", str(result))
        else:
            data = json.dumps(result, ensure_ascii=False, indent=2)

        return f"[工具调用成功] {tool_name}:\n{data}"

    def _truncate_if_needed(self, result: dict) -> dict:
        """截断过长的结果，保留关键信息。

        截断策略：
        1. 优先保留结构化字段（summary, key_points）
        2. 截断 data 字段而非丢弃
        3. 添加截断标记和原始长度信息
        """
        if result.get("raw"):
            data = result.get("data", "")
            if len(data) > self.max_result_length:
                result["data"] = (
                    data[:self.max_result_length] +
                    f"\n\n[结果被截断] 原始长度: {len(data)} 字符，"
                    f"显示前 {self.max_result_length} 字符。"
                    f"如需完整内容，请使用更精确的查询或分页参数。"
                )
        else:
            # 尝试截断 JSON 中的长字段
            for key in ["data", "results", "content", "body"]:
                if key in result and isinstance(result[key], str):
                    value = result[key]
                    if len(value) > self.max_result_length:
                        result[key] = value[:self.max_result_length] + "..."
                        result["_truncated"] = True
                        result["_original_length"] = len(value)

        return result


# 测试处理器
processor = ToolResultProcessor(max_result_length=200, max_retries=2)

print("=" * 60)
print("  工具结果处理测试")
print("=" * 60)

# 测试 1: 成功结果
print("\n[测试 1] 成功结果的格式化:")
success_result = json.dumps({
    "success": True,
    "city": "北京",
    "temperature": 22,
    "condition": "晴天",
})
formatted = processor.process("get_weather", {"city": "北京"}, success_result)
print(formatted)

# 测试 2: 错误结果
print("\n[测试 2] 错误结果的格式化:")
error_result = json.dumps({
    "success": False,
    "error": "除以零操作: 100/0",
    "error_type": "division_by_zero",
    "suggestion": "除数不能为零，请检查表达式",
    "retryable": True,
})
formatted = processor.process("calculator", {"expression": "100/0"}, error_result)
print(formatted)

# 测试 3: 过长结果截断
print("\n[测试 3] 过长结果的截断:")
long_result = "A" * 500
formatted = processor.process("web_search", {"query": "test"}, long_result)
print(formatted[:300] + "...")
print(f"\n格式化后长度: {len(formatted)} 字符")

### 重试逻辑实现

当工具返回可重试的错误时，Agent 应该智能地调整参数后重试，
而不是简单地用相同参数再试一次。

In [ ]:
# === 智能重试逻辑 ===

class RetryManager:
    """智能重试管理器。

    不是盲目重试，而是根据错误类型调整策略:
    - 除零错误 → 提示 LLM 修改表达式
    - 超时错误 → 缩小查询范围
    - 参数无效 → 修正参数格式
    - 无结果 → 扩展搜索范围
    """

    def __init__(self, max_retries: int = 2):
        self.max_retries = max_retries
        self.retry_history: list[dict] = []

    def should_retry(self, error: dict, attempt: int) -> bool:
        """判断是否应该重试。

        Args:
            error: 错误信息字典
            attempt: 当前尝试次数（从 0 开始）

        Returns:
            是否应该重试
        """
        if attempt >= self.max_retries:
            return False
        if not error.get("retryable", False):
            return False
        # 不可恢复的错误不重试
        non_recoverable = [
            "permission_denied", "auth_error", "quota_exceeded",
        ]
        if error.get("error_type") in non_recoverable:
            return False
        return True

    def adjust_params_for_retry(self, tool_name: str,
                                 original_params: dict,
                                 error: dict,
                                 attempt: int) -> dict:
        """根据错误类型智能调整参数。

        Args:
            tool_name: 工具名称
            original_params: 原始参数
            error: 错误信息
            attempt: 重试次数

        Returns:
            调整后的参数
        """
        adjusted = dict(original_params)
        error_type = error.get("error_type", "")
        suggestion = error.get("suggestion", "")

        if error_type == "division_by_zero":
            # 除零错误：无法自动修复，返回原参数
            # Agent 会收到错误提示并自行调整
            return adjusted

        elif error_type == "timeout":
            # 超时：缩小查询范围或减少结果数
            if "query" in adjusted:
                query = adjusted["query"]
                if len(query) > 50:
                    adjusted["query"] = query[:50]
            if "num_results" in adjusted:
                adjusted["num_results"] = max(1, adjusted["num_results"] // 2)

        elif error_type == "no_results":
            # 无结果：使用更通用的查询
            if "query" in adjusted:
                query = adjusted["query"]
                words = query.split()
                if len(words) > 3:
                    adjusted["query"] = " ".join(words[:3])

        elif error_type == "invalid_input":
            # 输入无效：去除特殊字符
            for key in adjusted:
                if isinstance(adjusted[key], str):
                    adjusted[key] = adjusted[key].strip().strip("'\"`")

        elif error_type == "out_of_range":
            # 参数超出范围：调整到边界值
            for key, value in adjusted.items():
                if isinstance(value, (int, float)):
                    if suggestion:
                        # 尝试从建议中提取范围
                        import re
                        range_match = re.search(
                            r'(\d+(?:\.\d+)?)\s*[-~到]\s*(\d+(?:\.\d+)?)',
                            suggestion)
                        if range_match:
                            lower = float(range_match.group(1))
                            upper = float(range_match.group(2))
                            mid = (lower + upper) / 2
                            adjusted[key] = type(value)(mid)

        # 记录重试历史
        self.retry_history.append({
            "tool": tool_name,
            "attempt": attempt,
            "original_params": original_params,
            "adjusted_params": adjusted,
            "error_type": error_type,
        })

        return adjusted

    def get_retry_summary(self) -> str:
        """生成重试历史摘要。"""
        if not self.retry_history:
            return "无重试记录"

        lines = [f"重试历史 (共 {len(self.retry_history)} 次):"]
        for entry in self.retry_history:
            lines.append(
                f"  [{entry['tool']}] 第 {entry['attempt']+1} 次重试 "
                f"({entry['error_type']}): "
                f"{entry['original_params']} → {entry['adjusted_params']}"
            )
        return "\n".join(lines)


# 测试重试管理器
retry_mgr = RetryManager(max_retries=2)

print("智能重试测试:")

# 测试 1: 超时错误 → 缩小查询
error_timeout = {
    "success": False,
    "error": "搜索超时",
    "error_type": "timeout",
    "retryable": True,
}
original = {"query": "Python programming language tutorial for beginners 2024", "num_results": 10}

print(f"\n原始参数: {original}")
for attempt in range(3):
    if retry_mgr.should_retry(error_timeout, attempt):
        adjusted = retry_mgr.adjust_params_for_retry(
            "web_search", original if attempt == 0 else adjusted,
            error_timeout, attempt)
        print(f"  重试 {attempt+1}: {adjusted}")
    else:
        print(f"  放弃重试（已尝试 {attempt} 次）")
        break

# 测试 2: 除零错误 → 不自动调整
error_divzero = {
    "success": False,
    "error": "除以零: 100/0",
    "error_type": "division_by_zero",
    "retryable": True,
}
original2 = {"expression": "100/0"}
print(f"\n除零错误 - 原始参数: {original2}")
adjusted2 = retry_mgr.adjust_params_for_retry(
    "calculator", original2, error_divzero, 0)
print(f"  参数保持不变（需 Agent 手动调整）: {adjusted2}")

print(f"\n{retry_mgr.get_retry_summary()}")

## 总结

### 本 Notebook 的核心要点

| 主题 | 关键知识点 | 实践技能 |
|------|-----------|---------|
| JSON Schema 设计 | 清晰的描述、参数约束、负面示例 | 编写生产级工具 Schema |
| 函数路由策略 | Auto/Forced/Pre-Classification 三种模式 | 根据场景选择最优路由策略 |
| 并行函数调用 | 依赖检测、批次分组、asyncio.gather | 实现并行工具执行器 |
| 工具结果处理 | 结构化错误、智能重试、截断、格式化 | 构建健壮的工具结果处理管道 |

### 最佳实践总结

1. **Schema 设计**：每个工具描述必须包含「使用场景」和「不要使用」
2. **路由策略**：工具 < 5 个用 Auto-Routing，5-15 个用 Pre-Classification，> 15 个分组路由
3. **并行执行**：先检测依赖再分组，批次内并行、批次间串行
4. **错误处理**：所有工具返回结构化 JSON，Agent 层智能重试和回退

### 下一步

现在你已经掌握了工具调用的核心技能，接下来阅读三个陷阱文档：
- `pitfalls/01-infinite-loop.md` —— 如何防止 Agent 无限循环
- `pitfalls/02-tool-call-failure.md` —— 如何处理工具调用失败
- `pitfalls/03-wrong-tool-selection.md` —— 如何避免工具选择错误